In [ ]:
#LangGraph Nodes, Edges & State

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class SimpleState(TypedDict):
    message: str

def greet(state: SimpleState) -> dict:
    return {"message": "Hello from the node!"}

graph = StateGraph(SimpleState)
graph.add_node("greet", greet)
graph.add_edge(START, "greet")
graph.add_edge("greet", END)

app = graph.compile()
result = app.invoke({"message": ""})
print(result)


In [ ]:
def transform_message(state: SimpleState) -> dict:
    original = state["message"]
    return {"message": original.upper() + " (transformed)"}

graph2 = StateGraph(SimpleState)
graph2.add_node("greet", greet)
graph2.add_node("transform", transform_message)
graph2.add_edge(START, "greet")
graph2.add_edge("greet", "transform")
graph2.add_edge("transform", END)

app2 = graph2.compile()
result2 = app2.invoke({"message": ""})
print(result2)


In [ ]:
#What Are Edges — And Why Do They Matter?

from langgraph.graph import START, END

# The minimum graph structure:
# START -> your_node -> END


In [ ]:
#Chaining Multiple Nodes

class PipelineState(TypedDict):
    text: str
    steps_completed: list[str]

def step_one(state: PipelineState) -> dict:
    return {
        "text": "raw data",
        "steps_completed": ["step_one"]
    }

def step_two(state: PipelineState) -> dict:
    updated = state["text"] + " -> cleaned"
    steps = state["steps_completed"] + ["step_two"]
    return {"text": updated, "steps_completed": steps}

def step_three(state: PipelineState) -> dict:
    updated = state["text"] + " -> analyzed"
    steps = state["steps_completed"] + ["step_three"]
    return {"text": updated, "steps_completed": steps}


In [ ]:
#

pipeline = StateGraph(PipelineState)
pipeline.add_node("step_one", step_one)
pipeline.add_node("step_two", step_two)
pipeline.add_node("step_three", step_three)

pipeline.add_edge(START, "step_one")
pipeline.add_edge("step_one", "step_two")
pipeline.add_edge("step_two", "step_three")
pipeline.add_edge("step_three", END)

app3 = pipeline.compile()
result3 = app3.invoke({"text": "", "steps_completed": []})
print(f"Final text: {result3['text']}")
print(f"Steps: {result3['steps_completed']}")


In [ ]:
#Defining State with TypedDict

from typing import TypedDict

class ChatState(TypedDict):
    user_input: str
    response: str
    turn_count: int


In [ ]:
class CounterState(TypedDict):
    value: int
    history: list[str]

def add_five(state: CounterState) -> dict:
    new_value = state["value"] + 5
    return {
        "value": new_value,
        "history": state["history"] + [f"add_five: {state['value']} -> {new_value}"]
    }

def double_it(state: CounterState) -> dict:
    new_value = state["value"] * 2
    return {
        "value": new_value,
        "history": state["history"] + [f"double_it: {state['value']} -> {new_value}"]
    }

def subtract_three(state: CounterState) -> dict:
    new_value = state["value"] - 3
    return {
        "value": new_value,
        "history": state["history"] + [f"subtract_three: {state['value']} -> {new_value}"]
    }


In [ ]:
counter_graph = StateGraph(CounterState)
counter_graph.add_node("add_five", add_five)
counter_graph.add_node("double_it", double_it)
counter_graph.add_node("subtract_three", subtract_three)

counter_graph.add_edge(START, "add_five")
counter_graph.add_edge("add_five", "double_it")
counter_graph.add_edge("double_it", "subtract_three")
counter_graph.add_edge("subtract_three", END)

counter_app = counter_graph.compile()
result4 = counter_app.invoke({"value": 10, "history": []})

print(f"Final value: {result4['value']}")
print("\nExecution trace:")
for entry in result4["history"]:
    print(f"  {entry}")
